# Module 12 Exercise (Starter): Zero-shot classification and retrieval with pretrained CLIP

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nsteve2407/llm-transformers-course/blob/master/notebooks/12-clip/exercise_starter.ipynb)

Module page: [Module 12: CLIP — Contrastive Vision-Language Pretraining](https://nsteve2407.github.io/llm-transformers-course/modules/12-clip/)

CLIP trains a dual encoder (image tower + text tower) with a symmetric contrastive loss over ~400M noisy
web image-caption pairs, projecting both modalities into one shared embedding space where cosine similarity
directly measures "does this image match this text?". The payoff is that classification and retrieval fall
out of the *same* frozen embedding space with **no task-specific head and no fine-tuning** -- you just
change what text you compare an image against. This notebook applies a real pretrained
`openai/clip-vit-base-patch32` checkpoint (no training here; the model's weights never change) to two tasks:

1. **Part A -- zero-shot classification on a labeled CIFAR-10 subset**: naive bare-class-name prompts vs. a
   single engineered template (`"a photo of a {class}"`) vs. **prompt ensembling** across ~8 diverse
   templates -- implemented the way the CLIP paper actually does it (average each template's *normalized*
   text embedding, then *renormalize* the average -- not average raw logits), with two concrete numerical
   correctness checks (unit-norm assertion, single-template-degenerates-exactly-to-baseline assertion).
2. **Part B -- retrieval over an unlabeled image corpus**: precompute CLIP image embeddings once for a
   broader CIFAR-10 sample, then implement `retrieve(query_text, k)` (text-to-image) and
   `retrieve_similar_images(query_image, k)` (image-to-image), plus an **open-vocabulary** demonstration
   with free-text queries describing concepts that are not any single one of CIFAR-10's original 10 labels.

## Design choices and judgment calls (documented up front)

- **`SMOKE_TEST` scope**: shrinks only the labeled-subset size (images/class) and the unlabeled retrieval
  corpus size. It never shrinks or swaps the model -- both modes load the exact same real pretrained
  `openai/clip-vit-base-patch32` weights and run every image through the real vision/text towers. There is
  no training anywhere in this notebook (CLIP is used purely for inference), so there are no epoch counts or
  learning rates to shrink either.
- **6 CIFAR-10 classes for Part A** (`airplane, automobile, bird, cat, dog, horse`) rather than all 10: a
  labeled subset large enough to make naive/engineered/ensembled accuracy differences visible, small enough
  to stay fast at full scale (this is pure inference -- no gradient steps -- so even the "full" setting is
  cheap, but keeping the class count modest keeps prompt-building and plotting readable).
- **CIFAR-10 images are 32x32**, far below the 224x224 resolution CLIP was pretrained on;
  `CLIPProcessor` resizes/upsamples them automatically. This blurs fine detail, but CLIP's zero-shot
  accuracy on CIFAR-10 is well known to still be high (the CLIP paper reports ~88-95% zero-shot on
  CIFAR-10 variants) because the *classification signal* here is coarse object category, not fine texture.
- **Prompt-ensembling convention (the correctness-critical part of this notebook)**: for each class, every
  template's text embedding is individually L2-normalized, the normalized embeddings are averaged, and
  *that average is renormalized to unit norm* before being used for cosine similarity. This is the CLIP
  paper's actual published convention (Appendix, "Prompt engineering and ensembling") -- averaging
  raw/unnormalized embeddings, or averaging per-template similarity scores instead, is a different (and
  non-standard) computation with different numerical behavior. Two concrete checks below make this
  verifiable rather than just asserted in prose: (a) every ensembled per-class embedding has unit L2 norm,
  and (b) ensembling over a single-element template list is numerically identical (`torch.allclose`) to the
  plain non-ensembled single-template case -- if renormalization or averaging were implemented wrong, this
  degeneracy would not hold exactly.
- **Retrieval corpus source**: a broader sample of `torchvision.datasets.CIFAR10`'s **train** split (kept
  separate from the **test**-split images used for Part A's labeled classification subset, so the two parts
  don't reuse the exact same images), spanning all 10 original CIFAR-10 classes. It is treated as fully
  **unlabeled** by `retrieve()`/`retrieve_similar_images()` -- true CIFAR-10 labels are kept only so this
  notebook can *print* human-readable annotations next to retrieved results and sanity-check them; the
  retrieval functions themselves never see or use those labels, which is exactly why they generalize to the
  open-vocabulary queries in Part B3 that have no CIFAR-10 label at all.
- **`get_image_features`/`get_text_features` + manual L2-normalization**, rather than calling
  `CLIPModel.forward` directly: this exposes L2-normalized per-modality embeddings as first-class values
  that get reused across naive/engineered/ensembled classification *and* both retrieval functions, instead
  of recomputing image embeddings from scratch for every prompt variant.

In [ ]:
try:
    import transformers
except ImportError:
    %pip install -q transformers

In [ ]:
%matplotlib inline
import os
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
import torchvision.datasets as datasets
from transformers import CLIPModel, CLIPProcessor

SMOKE_TEST = os.environ.get("SMOKE_TEST") == "1"
torch.manual_seed(0)
random.seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"SMOKE_TEST={SMOKE_TEST}, device={device}")

## Setup: load pretrained CLIP

`openai/clip-vit-base-patch32`: a ViT-B/32 image encoder + a Transformer text encoder, jointly pretrained
with a symmetric contrastive loss. Loaded once and used, frozen, for everything below.

In [ ]:
CHECKPOINT = "openai/clip-vit-base-patch32"
model = CLIPModel.from_pretrained(CHECKPOINT).to(device).eval()
processor = CLIPProcessor.from_pretrained(CHECKPOINT)

n_params = sum(p.numel() for p in model.parameters())
print(f"Loaded {CHECKPOINT} ({n_params / 1e6:.1f}M params) onto {device}")

# Concrete evidence this is the real pretrained checkpoint (151.3M params), not a random-init fallback.
assert n_params > 1.4e8, f"expected ~151M params for a real CLIP ViT-B/32 checkpoint, got {n_params}"

### Labeled CIFAR-10 subset (Part A)

A small subset of `torchvision.datasets.CIFAR10`'s **test** split, restricted to 6 of the 10 classes and
`N_PER_CLASS` images per class (deterministic: the first `N_PER_CLASS` test-set images of each class, by
dataset order).

In [ ]:
DATA_ROOT = "./data"
CIFAR10_CLASSES = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck",
]
SELECTED_CLASSES = ["airplane", "automobile", "bird", "cat", "dog", "horse"]
SELECTED_IDX = [CIFAR10_CLASSES.index(c) for c in SELECTED_CLASSES]
N_PER_CLASS = 2 if SMOKE_TEST else 25

cifar_test = datasets.CIFAR10(root=DATA_ROOT, train=False, download=True)
test_targets = np.array(cifar_test.targets)

eval_images, eval_labels = [], []
for local_idx, cls_idx in enumerate(SELECTED_IDX):
    positions = np.where(test_targets == cls_idx)[0][:N_PER_CLASS]
    for pos in positions.tolist():
        image, _ = cifar_test[pos]
        eval_images.append(image.convert("RGB"))
        eval_labels.append(local_idx)  # index into SELECTED_CLASSES, not the original CIFAR-10 label id

eval_labels = torch.tensor(eval_labels)
print(f"Labeled eval subset: {len(eval_images)} images across {len(SELECTED_CLASSES)} classes "
      f"({N_PER_CLASS}/class): {SELECTED_CLASSES}")

In [ ]:
n_show = min(6, len(SELECTED_CLASSES))
fig, axes = plt.subplots(1, n_show, figsize=(2.2 * n_show, 2.4))
shown = set()
for image, label in zip(eval_images, eval_labels.tolist()):
    if label in shown:
        continue
    axes[label].imshow(image)
    axes[label].set_title(SELECTED_CLASSES[label], fontsize=9)
    axes[label].axis("off")
    shown.add(label)
    if len(shown) == n_show:
        break
plt.suptitle("One example per class (labeled eval subset)")
plt.tight_layout()
plt.show()

## Part A: zero-shot classification -- naive vs. engineered vs. ensembled prompts

### Helper functions

`encode_texts`/`encode_images` call CLIP's text/image towers and L2-normalize the resulting embeddings
(`get_text_features`/`get_image_features` return the *projected* embeddings, unnormalized, so normalization
is done explicitly here rather than relying on any implicit behavior). `classify` scores L2-normalized
image embeddings against L2-normalized per-class text embeddings via cosine similarity, scaled by the
model's own learned `logit_scale` (exactly what `CLIPModel.forward` does internally to produce
`logits_per_image`), then takes the argmax over classes. Image embeddings for the eval subset are computed
**once** and reused across the naive/engineered/ensembled comparisons below, since none of the three
prompt variants changes what the *image* side of the embedding looks like -- only the text side changes.

In [ ]:
def encode_texts(prompts):
    """prompts: list[str]. Returns L2-normalized text embeddings, shape (len(prompts), proj_dim)."""
    inputs = processor(text=prompts, return_tensors="pt", padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        text_embeds = model.get_text_features(**inputs)
    return text_embeds / text_embeds.norm(dim=-1, keepdim=True)


def encode_images(images):
    """images: list[PIL.Image]. Returns L2-normalized image embeddings, shape (len(images), proj_dim)."""
    inputs = processor(images=images, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        image_embeds = model.get_image_features(**inputs)
    return image_embeds / image_embeds.norm(dim=-1, keepdim=True)


def classify(image_embeds, text_embeds_per_class):
    """image_embeds: (N, D) L2-normalized. text_embeds_per_class: (C, D) L2-normalized, one row per class.
    Returns (logits (N, C), preds (N,)) -- cosine similarity scaled by the model's learned logit_scale."""
    logit_scale = model.logit_scale.exp()
    logits = logit_scale * image_embeds @ text_embeds_per_class.t()
    preds = logits.argmax(dim=-1)
    return logits, preds


def accuracy(preds, labels):
    # .cpu() on both sides so this works regardless of whether preds ended up on cuda or cpu -- eval_labels
    # is a plain CPU tensor, while preds comes out of classify(), which runs on `device`.
    return (preds.cpu() == labels.cpu()).float().mean().item()


eval_image_embeds = encode_images(eval_images)  # computed once, reused for naive/engineered/ensembled below
print(f"eval_image_embeds: {tuple(eval_image_embeds.shape)}")

### A1. Naive prompts: the bare class name

The "prompt" for each class is just its name, e.g. `"cat"` -- no template, no context.

In [ ]:
naive_text_embeds = encode_texts(SELECTED_CLASSES)
naive_logits, naive_preds = classify(eval_image_embeds, naive_text_embeds)
naive_acc = accuracy(naive_preds, eval_labels)
print(f"Naive bare-class-name prompts: accuracy = {naive_acc:.3f} ({len(eval_images)} images)")

### A2. Engineered prompt: a single template

`"a photo of a {class}"` -- closer to the kind of sentence CLIP actually saw at pretraining time (web
alt-text captions), rather than an isolated word.

In [ ]:
ENGINEERED_TEMPLATE = "a photo of a {}"
engineered_prompts = [ENGINEERED_TEMPLATE.format(c) for c in SELECTED_CLASSES]
engineered_text_embeds = encode_texts(engineered_prompts)
engineered_logits, engineered_preds = classify(eval_image_embeds, engineered_text_embeds)
engineered_acc = accuracy(engineered_preds, eval_labels)
print(f"Engineered template ({ENGINEERED_TEMPLATE!r}): accuracy = {engineered_acc:.3f}")

### A3. Prompt ensembling -- average normalized embeddings, then renormalize

`ENSEMBLE_TEMPLATES` below is a representative subset (8 templates) of the style of ensemble the CLIP paper
uses (it publishes ~80 templates per dataset; a smaller diverse subset here is enough to demonstrate the
mechanism and see an accuracy effect). For each class:

1. Format every template with that class name and encode each one into an L2-normalized text embedding
   (`encode_texts`, one embedding per template).
2. **Average** the `T` normalized embeddings elementwise -> a single `(D,)` vector.
3. **Renormalize** that average to unit L2 norm (the averaged vector's norm is *not* generally 1, since
   averaging several different unit vectors shrinks the result toward the origin unless they all point in
   exactly the same direction).

This is the CLIP-paper convention: ensembling happens **in embedding space**, before cosine similarity is
computed -- not by averaging each template's raw similarity score/logit against the image after the fact.

In [ ]:
ENSEMBLE_TEMPLATES = [
    "a photo of a {}",
    "a blurry photo of a {}",
    "a photo of a small {}",
    "a photo of a large {}",
    "a photo of many {}",
    "a close-up photo of a {}",
    "a bright photo of a {}",
    "a low resolution photo of a {}",
]


def build_ensembled_text_embeddings(class_names, templates):
    """class_names: list[str]. templates: list[str], each containing one "{}" placeholder.
    For each class: encode every template into an L2-normalized text embedding, AVERAGE those embeddings
    across templates, then RE-NORMALIZE the averaged vector to unit L2 norm (the correct CLIP-paper
    convention -- average normalized per-template embeddings and renormalize, not average raw
    similarity scores/logits). Returns (len(class_names), proj_dim), each row unit L2 norm."""
    # TODO(starter): for each class in class_names, format every template with that class name, encode the
    # formatted prompts with encode_texts(...) to get one L2-normalized embedding per template, average
    # those embeddings across the template dimension (dim=0), then divide the average by its own
    # .norm() to renormalize it back to unit L2 norm. Stack one such renormalized vector per class
    # (torch.stack(..., dim=0)) and return the (len(class_names), proj_dim) result.
    raise NotImplementedError(
        "TODO: implement prompt-ensembling as average-then-renormalize: for each class, encode_texts(...) "
        "every formatted template into L2-normalized embeddings, mean_embed = per_template_embeds.mean(dim=0), "
        "then mean_embed = mean_embed / mean_embed.norm() to renormalize to unit L2 norm before stacking "
        "one row per class."
    )


ensembled_text_embeds = build_ensembled_text_embeddings(SELECTED_CLASSES, ENSEMBLE_TEMPLATES)
print(f"ensembled_text_embeds: {tuple(ensembled_text_embeds.shape)} "
      f"(from {len(ENSEMBLE_TEMPLATES)} templates x {len(SELECTED_CLASSES)} classes)")

### A4. Concrete correctness checks

Two numerical assertions, not just a prose claim:

- **(a) Unit L2 norm**: every ensembled per-class embedding must have L2 norm 1.0 (within float tolerance)
  *after* renormalization -- if step 3 above (renormalize the average) were skipped or done wrong, this
  would fail, since the raw average of several unit vectors is not itself unit-norm in general.
- **(b) Single-template degeneracy**: ensembling over a template list containing exactly one template must
  produce **exactly** (`torch.allclose`) the same embeddings as just encoding that one template directly
  (no ensembling at all) -- averaging a single vector with itself is a no-op, and renormalizing an
  already-unit-norm vector is also a no-op, so with `T=1` the "ensembled" and "non-ensembled" computations
  must coincide bit-for-bit (up to floating-point tolerance). This is a strong check: if averaging or
  renormalization were implemented with a bug that happens to not affect the *direction* of embeddings much
  for T>1 (e.g. a bug in *how* the average is taken), it can still show up as an exact mismatch here.

In [ ]:
# (a) Ensembled per-class embeddings must have unit L2 norm after renormalization.
ensembled_norms = ensembled_text_embeds.norm(dim=-1).cpu()
print(f"ensembled embedding norms: {[round(n, 6) for n in ensembled_norms.tolist()]}")
assert torch.allclose(ensembled_norms, torch.ones(len(SELECTED_CLASSES)), atol=1e-5), ensembled_norms
print("(a) ensembled per-class embeddings have unit L2 norm -- OK")

# (b) Ensembling with a single-template list must degenerate EXACTLY to the non-ensembled single-template case.
single_template = [ENSEMBLE_TEMPLATES[0]]
single_ensembled = build_ensembled_text_embeddings(SELECTED_CLASSES, single_template)
single_naive = encode_texts([ENSEMBLE_TEMPLATES[0].format(c) for c in SELECTED_CLASSES])
max_abs_diff = (single_ensembled - single_naive).abs().max().item()
print(f"max abs difference, single-template ensembled vs. non-ensembled: {max_abs_diff:.2e}")
assert torch.allclose(single_ensembled, single_naive, atol=1e-6), max_abs_diff
print("(b) single-template ensembling degenerates exactly to the non-ensembled case -- OK")

In [ ]:
ensembled_logits, ensembled_preds = classify(eval_image_embeds, ensembled_text_embeds)
ensembled_acc = accuracy(ensembled_preds, eval_labels)
print(f"Prompt-ensembled ({len(ENSEMBLE_TEMPLATES)} templates): accuracy = {ensembled_acc:.3f}")

### A5. Naive vs. engineered vs. ensembled: comparison

All three use the **exact same frozen model weights and the exact same image embeddings** -- the *only*
thing that changes between them is which text goes in on the class side. Any accuracy difference is
therefore entirely attributable to how well each prompt's text embedding lands near the true class's
semantic direction in the shared embedding space: a bare word like `"cat"` is a plausible sentence fragment
but not very close to the *distribution* of full-sentence alt-text captions CLIP was pretrained on, a single
well-chosen template is closer to that distribution, and averaging several templates (then renormalizing)
pulls the per-class embedding toward the direction that's *robust* across different ways of phrasing "this
is a {class}", averaging out any single template's idiosyncrasies. (With a small evaluation subset, the
exact ranking can be noisy run-to-run -- especially under `SMOKE_TEST`'s very small subset -- but the
mechanism explaining *why* they can differ holds regardless of the exact numbers below.)

In [ ]:
results = {
    "naive (bare class name)": naive_acc,
    "engineered (1 template)": engineered_acc,
    f"ensembled ({len(ENSEMBLE_TEMPLATES)} templates)": ensembled_acc,
}
for name, acc in results.items():
    print(f"{name:32s}: {acc:.3f}")

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(list(results.keys()), list(results.values()), color=["tab:gray", "tab:blue", "tab:green"])
ax.set_ylabel("accuracy")
ax.set_ylim(0, 1)
ax.set_title(f"Zero-shot CIFAR-10 subset accuracy ({len(SELECTED_CLASSES)} classes, n={len(eval_images)})")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()

if ensembled_acc >= max(naive_acc, engineered_acc):
    print("Ensembling matched or beat both single-prompt variants here.")
else:
    print("On this particular (small) subset, ensembling did not strictly beat every single-prompt variant "
          "-- with only a handful of images per class this is within normal run-to-run noise; the point of "
          "ensembling is robustness in aggregate/expectation, not a guaranteed win on every small sample.")

## Part B: retrieval over an unlabeled image corpus

### B1. Assemble the corpus and precompute image embeddings

**Source**: a broader sample of `torchvision.datasets.CIFAR10`'s **train** split (deliberately the *train*
split here, distinct from the *test*-split images used for Part A's labeled subset above), spanning all 10
original CIFAR-10 classes, `N_CORPUS_PER_CLASS` images per class. The corpus is treated as fully
**unlabeled**: `retrieve()`/`retrieve_similar_images()` below only ever see `corpus_images`/
`corpus_image_embeds`, never `corpus_labels_for_reference` -- that array is kept purely so this notebook can
*print* a human-readable label next to each retrieved result for sanity-checking, exactly the way one would
annotate results from a real unlabeled photo collection using known ground truth for evaluation purposes
only. Every corpus image's CLIP embedding is computed once, up front, and reused by every retrieval call
below (the whole point of precomputing an index).

In [ ]:
N_CORPUS_PER_CLASS = 3 if SMOKE_TEST else 20
cifar_train = datasets.CIFAR10(root=DATA_ROOT, train=True, download=True)
train_targets = np.array(cifar_train.targets)

corpus_images, corpus_labels_for_reference = [], []
for cls_idx in range(len(CIFAR10_CLASSES)):
    positions = np.where(train_targets == cls_idx)[0][:N_CORPUS_PER_CLASS]
    for pos in positions.tolist():
        image, label = cifar_train[pos]
        corpus_images.append(image.convert("RGB"))
        corpus_labels_for_reference.append(label)

print(f"Retrieval corpus: {len(corpus_images)} images spanning all {len(CIFAR10_CLASSES)} CIFAR-10 classes "
      f"({N_CORPUS_PER_CLASS}/class), from torchvision.datasets.CIFAR10's TRAIN split. Labels are kept only "
      f"for qualitative annotation below -- retrieve()/retrieve_similar_images() never use them.")

corpus_image_embeds = encode_images(corpus_images)  # precomputed once, reused by every retrieve() call below
print(f"corpus_image_embeds: {tuple(corpus_image_embeds.shape)}")

### B2. `retrieve(query_text, k)` and `retrieve_similar_images(query_image, k)`

Both rank the precomputed `corpus_image_embeds` by cosine similarity against a single query embedding
(text or image respectively) and return the top-`k` `(corpus_idx, score)` pairs, highest similarity first.
Since `corpus_image_embeds` and every embedding produced by `encode_texts`/`encode_images` are already
L2-normalized, a plain dot product *is* cosine similarity here -- no extra division needed.

In [ ]:
def retrieve(query_text, k=5):
    """query_text: str. k: int. Embeds query_text with CLIP's text encoder and ranks the precomputed
    corpus_image_embeds by cosine similarity. Returns list[(corpus_idx: int, score: float)], length k,
    highest similarity first."""
    # TODO(starter): embed query_text with encode_texts([query_text]) to get a (1, D) L2-normalized text
    # embedding, compute cosine similarity against every corpus embedding via
    # (text_embed @ corpus_image_embeds.t()).squeeze(0) (a plain dot product is cosine similarity here
    # since both sides are already L2-normalized), take the top k with .topk(k), and return a list of
    # (corpus_idx, score) pairs built from the returned indices/values.
    raise NotImplementedError(
        "TODO: text_embed = encode_texts([query_text]); sims = (text_embed @ corpus_image_embeds.t())."
        "squeeze(0); topk_scores, topk_idx = sims.topk(k); return list(zip(topk_idx.tolist(), "
        "topk_scores.tolist()))"
    )


def retrieve_similar_images(query_image, k=5):
    """query_image: PIL.Image. k: int. Embeds query_image with CLIP's image encoder and ranks the
    precomputed corpus_image_embeds by cosine similarity. Returns list[(corpus_idx: int, score: float)],
    length k, highest similarity first."""
    # TODO(starter): same pattern as retrieve() above, but embed the query with encode_images([query_image])
    # instead of encode_texts(...); everything else (cosine similarity against corpus_image_embeds via a
    # dot product, topk(k), zip into (idx, score) pairs) is identical.
    raise NotImplementedError(
        "TODO: query_embed = encode_images([query_image]); sims = (query_embed @ corpus_image_embeds.t())."
        "squeeze(0); topk_scores, topk_idx = sims.topk(k); return list(zip(topk_idx.tolist(), "
        "topk_scores.tolist()))"
    )


print("retrieve() and retrieve_similar_images() defined.")

In [ ]:
K = 5
demo_query = "a photo of a dog"
text_results = retrieve(demo_query, k=K)
print(f"retrieve({demo_query!r}, k={K}):")
for rank, (idx, score) in enumerate(text_results, 1):
    label = CIFAR10_CLASSES[corpus_labels_for_reference[idx]]
    print(f"  #{rank}: corpus idx {idx} (true CIFAR-10 label: {label}), cosine similarity = {score:.4f}")

fig, axes = plt.subplots(1, K, figsize=(3 * K, 3))
for ax, (idx, score) in zip(axes, text_results):
    ax.imshow(corpus_images[idx])
    ax.set_title(f"{CIFAR10_CLASSES[corpus_labels_for_reference[idx]]}\nsim={score:.3f}", fontsize=9)
    ax.axis("off")
plt.suptitle(f"retrieve({demo_query!r})")
plt.tight_layout()
plt.show()

# Well-formedness + sanity check: with 10 corpus classes, a random top-K would contain ~K/10 true "dog"
# images by chance; a working text-to-image retrieval should clearly beat that baseline for a query this
# unambiguous.
n_dog_hits = sum(1 for idx, _ in text_results if CIFAR10_CLASSES[corpus_labels_for_reference[idx]] == "dog")
chance = K / len(CIFAR10_CLASSES)
print(f"{n_dog_hits}/{K} retrieved images are actually labeled 'dog' (vs. {chance:.1f} expected by chance)")
assert len(text_results) == K
assert n_dog_hits >= 1, "expected at least one true 'dog' image in the top-K for an unambiguous text query"

In [ ]:
query_idx = 0
query_image = corpus_images[query_idx]
image_results = retrieve_similar_images(query_image, k=K)
print(f"retrieve_similar_images(corpus_images[{query_idx}], k={K}):")
for rank, (idx, score) in enumerate(image_results, 1):
    label = CIFAR10_CLASSES[corpus_labels_for_reference[idx]]
    marker = "  <- query image itself" if idx == query_idx else ""
    print(f"  #{rank}: corpus idx {idx} (true CIFAR-10 label: {label}), cosine similarity = {score:.4f}{marker}")

# Well-formedness + sanity check: querying with an image drawn directly from the corpus should return that
# exact image as the #1 result with cosine similarity ~1.0 (self-similarity, the strongest possible signal
# that image-to-image retrieval is actually comparing embeddings correctly).
top1_idx, top1_score = image_results[0]
assert len(image_results) == K
assert top1_idx == query_idx, f"expected the query image itself as the top-1 result, got idx {top1_idx}"
assert top1_score > 0.99, f"expected near-1.0 self-similarity, got {top1_score}"
print("image-to-image retrieval sanity check passed: querying with a corpus image returns itself as the "
      "top-1 result with cosine similarity ~1.0.")

### B3. Open-vocabulary demonstration

CIFAR-10's original label set is exactly 10 fixed category names. The three queries below are
**compositional descriptions that are not any single one of those 10 labels**, to demonstrate that
`retrieve()` works over arbitrary free text, not just a fixed vocabulary it was somehow tuned for -- there is
no classifier head or label list involved anywhere in `retrieve()`; it is the same general
text-to-image cosine-similarity search used for `"a photo of a dog"` above, just with different text:

- `"a red vehicle"` -- composes a color attribute with a coarse category that spans several CIFAR-10 classes
  (automobile, truck, airplane, ship can all be "vehicles"; only some instances of each are red).
- `"an animal with fur"` -- composes an attribute with a coarse category that spans several CIFAR-10 classes
  (cat, dog, deer, horse have fur; bird and frog do not).
- `"a black and white animal"` -- composes a color-pattern attribute with the same coarse animal category,
  picking out a *different* cross-class subset than the fur query.

For each, the retrieved images and their true CIFAR-10 labels (shown only as an external cross-check, never
used by `retrieve()` itself) are printed and plotted, and the label distribution among the top-k is reported
to show the results are **not** confined to a single fixed class.

In [ ]:
OPEN_VOCAB_QUERIES = [
    "a red vehicle",
    "an animal with fur",
    "a black and white animal",
]

for query in OPEN_VOCAB_QUERIES:
    results = retrieve(query, k=K)
    labels_seen = [CIFAR10_CLASSES[corpus_labels_for_reference[idx]] for idx, _ in results]
    print(f"retrieve({query!r}, k={K}) -> labels: {labels_seen}")

    fig, axes = plt.subplots(1, K, figsize=(3 * K, 3))
    for ax, (idx, score) in zip(axes, results):
        ax.imshow(corpus_images[idx])
        ax.set_title(f"{CIFAR10_CLASSES[corpus_labels_for_reference[idx]]}\nsim={score:.3f}", fontsize=9)
        ax.axis("off")
    plt.suptitle(f"open-vocabulary retrieve({query!r})")
    plt.tight_layout()
    plt.show()

    n_unique_labels = len(set(labels_seen))
    print(f"  -> {n_unique_labels} distinct CIFAR-10 label(s) among the top-{K} results: {sorted(set(labels_seen))}\n")

## Summary

**What zero-shot + open-vocabulary buys over a fixed-head classifier**: a conventional classifier head is a
fixed matrix baked in at training time -- it can only ever emit scores over the exact label set it was
trained on, and adding a new class means retraining (or at least re-fitting) that head. Every CLIP
"classifier" used above (naive, engineered, ensembled) is instead just a set of text embeddings computed on
the fly from arbitrary text against the same frozen image tower -- Part A's 6-way classifier and Part B3's
free-text queries used the *exact same* weights, with zero retraining in between. Retrieval in Part B makes
this maximally concrete: `retrieve()` never sees a label list at all, so it works identically well whether
the query happens to match one of CIFAR-10's 10 original classes or describes an attribute/composition
("a red vehicle", "an animal with fur") that cuts *across* several of them.

**Where this specific setup would show CLIP's known limitations** (per the module reading list):

- **Fine-grained recognition**: CLIP's contrastive pretraining rewards distinguishing *broad* semantic
  categories (the kind of distinction alt-text captions usually make), not fine-grained distinctions within
  a category (e.g. telling apart specific dog breeds, or specific car models) -- nothing here tests that,
  but it's a well-documented weak point relative to models fine-tuned specifically for a fine-grained task.
- **Counting**: none of the prompts above ask "how many" of anything; CLIP is known to be weak at
  numerosity, since a caption like "three dogs" and "two dogs" are extremely close to each other, and to
  "some dogs", in embedding space, unlike a model with an explicit counting mechanism.
- **Spatial reasoning**: none of the open-vocabulary queries encode relative position (e.g. "a cat to the
  left of a dog"); CLIP's embedding space is known to be much better at *what* is in an image than *where*
  things are relative to each other, since bag-of-concepts-style captions dominate its training data far
  more than spatially precise ones.
- **Typographic attacks**: pasting text of a *different* class name onto/near an object is well documented
  to shift CLIP's prediction toward the pasted word rather than the actual depicted object, because CLIP's
  text and image encoders share one embedding space and CLIP was never trained to distinguish "an image
  containing this word" from "an image whose caption would contain this word." Nothing in this notebook
  exercises that (CIFAR-10 photos don't contain overlaid text), but it follows directly from the same
  shared-embedding-space mechanism that makes Part B3's open-vocabulary retrieval work in the first place --
  the mechanism has no way to tell "text describing the image" apart from "text visible inside the image."